# 03 - Fraud Pattern Analysis

This notebook performs in-depth fraud pattern detection and analysis.

## Objectives
- Detect fraud patterns across all entities (drivers, customers, regions)
- Analyze regional and temporal fraud patterns
- Analyze product categories most affected by fraud
- Generate risk scores and rankings
- Produce actionable fraud reports

## Data Source
- PostgreSQL database: `walmart-delivery-fraud-detection`
- Schema: `walmart_fraud`
- Period: January 2023 - December 2023
- Region: Central Florida

In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

import sys
sys.path.insert(0, '..')

# Visualization theme (MUST be imported first)
from src.config.viz_theme import (
    PROJECT_THEME, REGION_COLORS, 
    get_highlight_colors, apply_project_theme, get_label,
    get_top_n_highlight_colors
)

# Database connection - PostgreSQL
from src.database.connection import (
    load_orders, 
    load_drivers, 
    load_customers, 
    load_products, 
    load_missing_items, 
    test_connection
)

# Feature engineering modules
from src.features.driver_features import create_driver_features, get_driver_risk_scores
from src.features.customer_features import create_customer_features, get_customer_risk_scores

# Analysis modules
from src.analysis.fraud_patterns import (
    analyze_all_fraud_patterns, 
    generate_fraud_report,
    detect_collusion_patterns
)
from src.analysis.geographic import analyze_regional_performance, identify_regional_hotspots
from src.analysis.temporal import get_temporal_summary, detect_temporal_anomalies

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
%matplotlib inline

# Plotly default template
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [3]:
# Test database connection
if test_connection():
    print('Database connection: OK')
else:
    raise Exception('Database connection failed!')

Database connection: OK


In [4]:
# Load all data from PostgreSQL
orders = load_orders()
drivers = load_drivers()
customers = load_customers()
products = load_products()
missing_items = load_missing_items()

print('Data loaded successfully!')
print(f'  Orders: {len(orders):,} records')
print(f'  Drivers: {len(drivers):,} records')
print(f'  Customers: {len(customers):,} records')
print(f'  Products: {len(products):,} records')
print(f'  Missing Items: {len(missing_items):,} records')

Data loaded successfully!
  Orders: 10,000 records
  Drivers: 1,247 records
  Customers: 1,239 records
  Products: 314 records
  Missing Items: 1,662 records


## 1. Overall Fraud Metrics

In [5]:
# Calculate overall metrics
total_orders = len(orders)
total_items = orders['items_delivered'].sum() + orders['items_missing'].sum()
total_missing = orders['items_missing'].sum()
orders_with_missing = (orders['items_missing'] > 0).sum()
overall_missing_rate = total_missing / total_items * 100
pct_orders_with_missing = orders_with_missing / total_orders * 100

# Estimated loss calculation (using average product price)
avg_product_price = products['price'].mean()
estimated_loss = total_missing * avg_product_price

print('=' * 50)
print('OVERALL FRAUD METRICS')
print('=' * 50)
print(f'Total Orders: {total_orders:,}')
print(f'Total Items: {total_items:,}')
print(f'Total Missing Items: {total_missing:,}')
print(f'Overall Missing Rate: {overall_missing_rate:.2f}%')
print(f'Orders with Missing Items: {orders_with_missing:,} ({pct_orders_with_missing:.2f}%)')
print(f'Average Product Price: ${avg_product_price:.2f}')
print(f'Estimated Loss: ${estimated_loss:,.2f}')

OVERALL FRAUD METRICS
Total Orders: 10,000
Total Items: 101,351
Total Missing Items: 1,657
Overall Missing Rate: 1.63%
Orders with Missing Items: 1,502 (15.02%)
Average Product Price: $59.13
Estimated Loss: $97,978.30


In [6]:
# Fraud metrics visualization with context
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}]],
    subplot_titles=['Overall Missing Rate', 'Orders with Issues']
)

fig.add_trace(
    go.Indicator(
        mode='gauge+number',
        value=overall_missing_rate,
        number={'suffix': '%', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 30]},
            'bar': {'color': PROJECT_THEME['walmart_blue']},
            'steps': [
                {'range': [0, 10], 'color': PROJECT_THEME['risk_colors']['Low']},
                {'range': [10, 20], 'color': PROJECT_THEME['risk_colors']['Medium']},
                {'range': [20, 30], 'color': PROJECT_THEME['risk_colors']['Critical']}
            ],
            'threshold': {
                'line': {'color': 'black', 'width': 4},
                'thickness': 0.75,
                'value': overall_missing_rate
            }
        }
    ),
    row=1, col=1
)

fig.add_trace(
    go.Indicator(
        mode='gauge+number',
        value=pct_orders_with_missing,
        number={'suffix': '%', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 50]},
            'bar': {'color': PROJECT_THEME['walmart_blue']},
            'steps': [
                {'range': [0, 15], 'color': PROJECT_THEME['risk_colors']['Low']},
                {'range': [15, 30], 'color': PROJECT_THEME['risk_colors']['Medium']},
                {'range': [30, 50], 'color': PROJECT_THEME['risk_colors']['Critical']}
            ],
            'threshold': {
                'line': {'color': 'black', 'width': 4},
                'thickness': 0.75,
                'value': pct_orders_with_missing
            }
        }
    ),
    row=1, col=2
)

# Add annotations explaining thresholds
fig.add_annotation(
    text='<b>Thresholds:</b> <10% Low | 10-20% Medium | >20% Critical',
    xref='paper', yref='paper',
    x=0.25, y=-0.15,
    showarrow=False,
    font=dict(size=10, color=PROJECT_THEME['neutral_gray']),
    xanchor='center'
)

fig.add_annotation(
    text='<b>Thresholds:</b> <15% Low | 15-30% Medium | >30% Critical',
    xref='paper', yref='paper',
    x=0.75, y=-0.15,
    showarrow=False,
    font=dict(size=10, color=PROJECT_THEME['neutral_gray']),
    xanchor='center'
)

fig.update_layout(
    title='Fraud Exposure Metrics - Performance vs Industry Benchmarks',
    height=400,
    margin=dict(b=60)
)

fig = apply_project_theme(fig)
fig.show()

## 2. Comprehensive Fraud Pattern Detection

In [7]:
# Run all fraud pattern analyses
fraud_indicators = analyze_all_fraud_patterns(orders, drivers, customers)

print('Fraud Indicators Found:')
for pattern_type, indicators in fraud_indicators.items():
    print(f'  - {pattern_type}: {len(indicators)} indicators')

Fraud Indicators Found:
  - driver_patterns: 56 indicators
  - customer_patterns: 108 indicators
  - collusion_patterns: 0 indicators
  - regional_patterns: 0 indicators


In [8]:
# Generate fraud report
fraud_report = generate_fraud_report(fraud_indicators)

print(f"\nTotal Fraud Indicators: {fraud_report['summary']['total_indicators']}")
print(f"High Risk: {len(fraud_report['high_risk'])}")
print(f"Medium Risk: {len(fraud_report['medium_risk'])}")
print(f"Low Risk: {len(fraud_report['low_risk'])}")


Total Fraud Indicators: 164
High Risk: 0
Medium Risk: 58
Low Risk: 106


In [9]:
# Visualize fraud indicators by type
indicator_counts = pd.DataFrame([
    {'Type': k.replace('_', ' ').title(), 'Count': len(v)} 
    for k, v in fraud_indicators.items()
])

fig = px.bar(
    indicator_counts,
    x='Type',
    y='Count',
    title='Fraud Indicators by Type',
    color='Count',
    color_continuous_scale='Reds',
    text='Count',
    labels={'Count': 'Number of Indicators', 'Type': 'Indicator Type'}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=400)
fig = apply_project_theme(fig)
fig.show()

In [10]:
# Display high-risk entities
if fraud_report['high_risk']:
    high_risk_df = pd.DataFrame(fraud_report['high_risk'])
    print('HIGH RISK ENTITIES')
    print('=' * 50)
    display(high_risk_df[['entity_type', 'entity_id', 'indicator', 'score']].head(10))
else:
    print('No high-risk entities detected')

No high-risk entities detected


## 3. Driver Risk Analysis

In [11]:
# Create driver features and risk scores
driver_features = create_driver_features(drivers, orders)
driver_risk = get_driver_risk_scores(driver_features)

print(f'Driver features created for {len(driver_risk)} drivers')
print(f'Columns: {list(driver_risk.columns)}')

Driver features created for 1247 drivers
Columns: ['driver_id', 'driver_name', 'age', 'trips', 'total_orders', 'total_items_delivered', 'total_items_missing', 'total_revenue', 'avg_order_value', 'missing_rate', 'orders_with_missing', 'pct_orders_with_missing', 'age_group', 'experience_level', 'missing_rate_score', 'pct_orders_score', 'volume_score', 'risk_score', 'risk_category']


In [12]:
# Driver risk distribution
fig = px.histogram(
    driver_risk,
    x='risk_score',
    color='risk_category',
    nbins=20,
    title='Driver Risk Score Distribution',
    labels={
        'risk_score': get_label('risk_score'), 
        'risk_category': 'Risk Category',
        'count': 'Number of Drivers'
    },
    color_discrete_map=PROJECT_THEME['risk_colors'],
    category_orders={'risk_category': PROJECT_THEME['risk_order_asc']}
)
fig.update_layout(height=400, bargap=0.1)
fig = apply_project_theme(fig)
fig.show()

In [13]:
# Top 10 highest risk drivers
top_risk_drivers = driver_risk.nlargest(10, 'risk_score')[[
    'driver_id', 'driver_name', 'age', 'total_orders', 
    'missing_rate', 'risk_score', 'risk_category'
]].copy()

print('TOP 10 HIGHEST RISK DRIVERS')
print('=' * 50)
display(top_risk_drivers)

TOP 10 HIGHEST RISK DRIVERS


,driver_id,driver_name,age,total_orders,missing_rate,risk_score,risk_category
450,WDID10322,Dana Ferguson,22,11,7.46,95.83,Critical
3,WDID09875,Stephen Randolph,18,11,6.52,94.96,Critical
12,WDID09884,James Winters,18,11,6.52,94.96,Critical
350,WDID10222,Daniel Hall,20,11,7.04,93.58,Critical
10,WDID09882,Angela Young,18,11,6.12,92.82,Critical
9,WDID09881,Tracey Watkins,18,11,6.06,92.48,Critical
1,WDID09873,Pamela Moore,18,11,6.00,92.16,Critical
7,WDID09879,Shane Palmer,18,11,5.94,91.84,Critical
575,WDID10447,Patricia Vance,18,11,6.49,90.64,Critical
4,WDID09876,Jordan Daniel,18,11,5.71,90.63,Critical


In [14]:
# Risk category breakdown
risk_counts = driver_risk['risk_category'].value_counts().reset_index()
risk_counts.columns = ['Risk Category', 'Count']

# Count high risk drivers
high_risk_drivers_count = len(driver_risk[driver_risk['risk_category'].isin(['High', 'Critical'])])

fig = px.pie(
    risk_counts,
    values='Count',
    names='Risk Category',
    title='Driver Risk Category Distribution',
    color='Risk Category',
    color_discrete_map=PROJECT_THEME['risk_colors'],
    hole=0.4,
    category_orders={'Risk Category': PROJECT_THEME['risk_order']}
)
fig.update_traces(textposition='outside', textinfo='percent+label+value')
fig.update_layout(height=400)
fig = apply_project_theme(fig)
fig.show()

print(f'\nHigh/Critical Risk Drivers: {high_risk_drivers_count}')


High/Critical Risk Drivers: 419


In [15]:
# Driver risk scatter plot
fig = px.scatter(
    driver_risk[driver_risk['total_orders'] > 0],
    x='total_orders',
    y='missing_rate',
    color='risk_category',
    size='total_items_missing',
    hover_data=['driver_id', 'driver_name', 'risk_score'],
    title='Driver Risk Analysis: Orders vs Missing Rate',
    labels={
        'total_orders': get_label('total_orders'),
        'missing_rate': get_label('missing_rate'),
        'risk_category': 'Risk Category',
        'total_items_missing': 'Items Missing'
    },
    color_discrete_map=PROJECT_THEME['risk_colors'],
    category_orders={'risk_category': PROJECT_THEME['risk_order_asc']}
)
fig.update_layout(height=500)
fig = apply_project_theme(fig)
fig.show()

## 4. Customer Risk Analysis

In [16]:
# Create customer features and risk scores
customer_features = create_customer_features(customers, orders)
customer_risk = get_customer_risk_scores(customer_features)

print(f'Customer features created for {len(customer_risk)} customers')

Customer features created for 1239 customers


In [17]:
# Customer risk distribution (only customers with orders)
active_customers = customer_risk[customer_risk['total_orders'] > 0]

fig = px.histogram(
    active_customers,
    x='risk_score',
    color='risk_category',
    nbins=20,
    title='Customer Risk Score Distribution',
    labels={
        'risk_score': get_label('risk_score'),
        'risk_category': 'Risk Category',
        'count': 'Number of Customers'
    },
    color_discrete_map=PROJECT_THEME['risk_colors'],
    category_orders={'risk_category': PROJECT_THEME['risk_order_asc']}
)
fig.update_layout(height=400, bargap=0.1)
fig = apply_project_theme(fig)
fig.show()

In [18]:
# Top 10 highest risk customers (with at least 2 orders)
top_risk_customers = customer_risk[customer_risk['total_orders'] >= 2].nlargest(10, 'risk_score')[[
    'customer_id', 'customer_name', 'customer_age', 'total_orders',
    'missing_rate', 'total_spent', 'risk_score', 'risk_category'
]].copy()

# Count high risk customers
high_risk_customers_count = len(customer_risk[
    (customer_risk['total_orders'] >= 2) & 
    (customer_risk['risk_category'].isin(['High', 'Critical']))
])

print('TOP 10 HIGHEST RISK CUSTOMERS')
print('=' * 50)
display(top_risk_customers)

TOP 10 HIGHEST RISK CUSTOMERS


,customer_id,customer_name,customer_age,total_orders,missing_rate,total_spent,risk_score,risk_category
743,WCID5743,Justin Williams,78,4,13.64,917.03,64.64,High
1043,WCID6043,Joshua Thompson,79,3,11.54,702.76,64.32,High
698,WCID5698,Crystal Smith,44,4,8.89,777.93,63.04,High
685,WCID5685,Shelley Gould,85,12,4.93,2361.59,59.88,High
711,WCID5711,Matthew Barker,46,3,9.52,970.23,58.41,High
820,WCID5820,Edward Henry,73,15,5.26,5071.83,56.77,High
923,WCID5923,Kenneth Brown,19,7,7.14,1224.29,55.24,High
762,WCID5762,Matthew Osborne,74,7,10.00,2384.49,55.05,High
672,WCID5672,Dana Gutierrez,19,7,6.67,2115.65,53.84,High
165,WCID5165,David Mccoy,59,7,6.59,2031.42,53.63,High


In [19]:
# Customer risk by spending segment
segment_risk = customer_risk[customer_risk['total_orders'] > 0].groupby('customer_segment').agg({
    'risk_score': 'mean',
    'missing_rate': 'mean',
    'customer_id': 'count'
}).reset_index()
segment_risk.columns = ['Segment', 'Avg Risk Score', 'Avg Missing Rate', 'Customer Count']

fig = px.bar(
    segment_risk,
    x='Segment',
    y='Avg Risk Score',
    color='Avg Missing Rate',
    title='Average Risk Score by Customer Segment',
    labels={
        'Avg Risk Score': 'Average Risk Score (0-100)',
        'Avg Missing Rate': 'Average Missing Rate (%)',
        'Segment': 'Customer Segment'
    },
    color_continuous_scale='Reds',
    text='Customer Count'
)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.update_layout(height=400)
fig = apply_project_theme(fig)
fig.show()

## 5. Regional Fraud Hotspots

In [20]:
# Regional performance
regional = analyze_regional_performance(orders)

# Identify worst region
worst_region = regional.iloc[0]['region']
worst_region_rate = regional.iloc[0]['missing_rate']

print('REGIONAL PERFORMANCE (Sorted by Missing Rate)')
print('=' * 50)
display(regional[['region', 'total_orders', 'missing_rate', 'items_missing', 'pct_orders_with_missing']])

REGIONAL PERFORMANCE (Sorted by Missing Rate)


,region,total_orders,missing_rate,items_missing,pct_orders_with_missing
0,Altamonte Springs,1426,1.76,253,16.20
1,Apopka,1422,1.73,249,15.33
2,Clermont,1384,1.72,243,15.82
4,Orlando,1401,1.64,233,15.13
6,Winter Park,1485,1.56,235,14.48
5,Sanford,1461,1.54,225,13.89
3,Kissimmee,1421,1.50,219,14.36


In [21]:
# Identify hotspots
hotspots = identify_regional_hotspots(orders, threshold_pct=75)

print('Regional Hotspots:')
for category, regions in hotspots.items():
    print(f'  {category}: {regions}')

Regional Hotspots:
  high_missing_rate: ['Altamonte Springs', 'Apopka']
  high_volume_missing: ['Altamonte Springs', 'Apopka']
  low_driver_coverage: ['Winter Park', 'Sanford']


In [22]:
# Regional missing rate horizontal bar chart
avg_regional_rate = regional['missing_rate'].mean()

# Sort data for visualization
regional_sorted = regional.sort_values('missing_rate', ascending=True).copy()

# Create highlight colors - worst region (last in sorted order) gets highlighted
regional_sorted['color'] = get_top_n_highlight_colors(
    regional_sorted, 
    top_n=1, 
    neutral=PROJECT_THEME['neutral_gray'],
    highlight=PROJECT_THEME['fraud_red'],
    ascending=False
)

fig = px.bar(
    regional_sorted,
    x='missing_rate',
    y='region',
    orientation='h',
    color='color',
    color_discrete_map='identity',
    title='Missing Rate by Region (Fraud Risk Indicator)',
    labels={
        'missing_rate': get_label('missing_rate'),
        'region': 'Region'
    },
    text='missing_rate'
)

# Add average reference line
fig.add_vline(
    x=avg_regional_rate,
    line_dash='dash',
    line_color=PROJECT_THEME['walmart_blue'],
    annotation_text=f'Average: {avg_regional_rate:.2f}%',
    annotation_position='top'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside', showlegend=False)
fig.update_layout(height=450)
fig = apply_project_theme(fig)
fig.show()

In [23]:
# Regional comparison: Orders vs Missing Items
fig = px.scatter(
    regional,
    x='total_orders',
    y='items_missing',
    size='missing_rate',
    color='missing_rate',
    text='region',
    title='Regional Analysis: Order Volume vs Missing Items',
    labels={
        'total_orders': get_label('total_orders'),
        'items_missing': 'Items Missing (count)',
        'missing_rate': get_label('missing_rate')
    },
    color_continuous_scale='Reds'
)
fig.update_traces(textposition='top center')
fig.update_layout(height=500)
fig = apply_project_theme(fig)
fig.show()

## 6. Temporal Fraud Patterns

In [24]:
# Temporal summary
temporal_summary = get_temporal_summary(orders)

print('Temporal Fraud Analysis:')
print(f"  Trend Direction: {temporal_summary['trend']['direction']}")
print(f"  Peak Month: {temporal_summary['trend']['peak_month']} ({temporal_summary['trend']['peak_rate']:.2f}%)")
print(f"  Lowest Month: {temporal_summary['trend']['lowest_month']} ({temporal_summary['trend']['lowest_rate']:.2f}%)")
print(f"  Worst Day: {temporal_summary['patterns']['worst_day']}")
print(f"  Worst Hour: {temporal_summary['patterns']['worst_hour']}:00")
print(f"  Weekend vs Weekday Rate: {temporal_summary['patterns']['weekend_vs_weekday']['weekend_rate']:.2f}% vs {temporal_summary['patterns']['weekend_vs_weekday']['weekday_rate']:.2f}%")

Temporal Fraud Analysis:
  Trend Direction: stable
  Peak Month: 2023-08 (1.83%)
  Lowest Month: 2023-02 (1.42%)
  Worst Day: Monday
  Worst Hour: 22:00
  Weekend vs Weekday Rate: 1.64% vs 1.63%


In [25]:
# Temporal anomalies
anomalies = detect_temporal_anomalies(orders)

print('\nTemporal Anomalies Detected:')
for anomaly_type, items in anomalies.items():
    if items:
        print(f'  {anomaly_type}: {len(items)} anomalies')
        for item in items[:3]:  # Show first 3
            print(f'    - {item}')


Temporal Anomalies Detected:


In [26]:
# Monthly trend visualization
orders_temp = orders.copy()
orders_temp['order_date'] = pd.to_datetime(orders_temp['order_date'])
orders_temp['month'] = orders_temp['order_date'].dt.month

monthly = orders_temp.groupby('month').agg({
    'order_id': 'count',
    'items_delivered': 'sum',
    'items_missing': 'sum'
}).reset_index()
monthly['missing_rate'] = monthly['items_missing'] / (monthly['items_delivered'] + monthly['items_missing']) * 100
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly['month_name'] = monthly['month'].apply(lambda x: month_names[x-1])

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=monthly['month_name'], 
        y=monthly['order_id'], 
        name='Orders', 
        marker_color=PROJECT_THEME['walmart_blue']
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=monthly['month_name'], 
        y=monthly['missing_rate'], 
        name='Missing Rate', 
        mode='lines+markers', 
        marker_color=PROJECT_THEME['fraud_red'], 
        line=dict(width=3)
    ),
    secondary_y=True
)

fig.update_layout(title='Monthly Fraud Trend (2023)', height=450)
fig.update_xaxes(title_text='Month')
fig.update_yaxes(title_text='Number of Orders', secondary_y=False)
fig.update_yaxes(title_text='Missing Rate (%)', secondary_y=True)
fig = apply_project_theme(fig)
fig.show()

## 7. Product Category Analysis

In [27]:
# Analyze missing items by product category
print('Missing Items Analysis by Category')
print('=' * 50)

# Missing items already has category info from database join
if 'category' in missing_items.columns:
    category_stats = missing_items.groupby('category').agg({
        'missing_item_id': 'count',
        'product_price': ['sum', 'mean']
    }).reset_index()
    category_stats.columns = ['category', 'missing_count', 'total_loss', 'avg_price']
    category_stats = category_stats.sort_values('missing_count', ascending=False)
    
    print(f'Categories analyzed: {len(category_stats)}')
    display(category_stats)
else:
    print('Category information not available in missing_items data')
    category_stats = None

Missing Items Analysis by Category
Categories analyzed: 2


,category,missing_count,total_loss,avg_price
1,Supermarket,1393,15974.01,11.47
0,Electronics,269,133398.60,495.91


In [28]:
# Product category fraud visualization
if category_stats is not None and len(category_stats) > 0:
    # Bar chart of missing items by category
    fig = px.bar(
        category_stats.sort_values('missing_count', ascending=True),
        x='missing_count',
        y='category',
        orientation='h',
        color='total_loss',
        color_continuous_scale='Reds',
        title='Missing Items by Product Category',
        labels={
            'missing_count': 'Number of Missing Items', 
            'category': 'Product Category', 
            'total_loss': 'Total Loss ($)'
        },
        text='missing_count'
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500)
    fig = apply_project_theme(fig)
    fig.show()

In [29]:
# Category loss analysis
if category_stats is not None and len(category_stats) > 0:
    fig = px.treemap(
        category_stats,
        path=['category'],
        values='total_loss',
        color='avg_price',
        color_continuous_scale='RdYlGn_r',
        title='Financial Loss by Product Category (Size = Total Loss, Color = Avg Price)',
        labels={
            'total_loss': 'Total Loss ($)', 
            'avg_price': 'Average Price ($)'
        }
    )
    fig.update_layout(height=500)
    fig = apply_project_theme(fig)
    fig.show()
    
    # Identify highest loss category
    highest_loss_category = category_stats.iloc[0]['category']
    highest_loss_amount = category_stats['total_loss'].max()
    print(f'\nHighest Loss Category: {highest_loss_category} (${highest_loss_amount:,.2f})')


Highest Loss Category: Supermarket ($133,398.60)


In [30]:
# Top 10 most frequently missing products
if 'product_name' in missing_items.columns:
    top_missing_products = missing_items.groupby(['product_id', 'product_name', 'category']).agg({
        'missing_item_id': 'count',
        'product_price': 'first'
    }).reset_index()
    top_missing_products.columns = ['product_id', 'product_name', 'category', 'times_missing', 'price']
    top_missing_products = top_missing_products.sort_values('times_missing', ascending=False).head(10)
    
    print('\nTOP 10 MOST FREQUENTLY MISSING PRODUCTS')
    print('=' * 50)
    display(top_missing_products)


TOP 10 MOST FREQUENTLY MISSING PRODUCTS


,product_id,product_name,category,times_missing,price
15,PWPX0982761090997,Chicken Breast,Supermarket,19,5.93
11,PWPX0982761090993,Ground Coffee,Supermarket,19,13.28
5,PWPX0982761090987,Cheddar Cheese,Supermarket,18,12.47
10,PWPX0982761090992,Canned Tuna,Supermarket,17,18.60
1,PWPX0982761090983,Uncured Bacon,Supermarket,17,4.67
12,PWPX0982761090994,Peanut Butter,Supermarket,17,19.32
96,PWPX0982761091078,Frozen Shrimp,Supermarket,17,4.71
31,PWPX0982761091013,Rice Cakes,Supermarket,16,6.56
72,PWPX0982761091054,Turkey Burgers,Supermarket,16,8.48
14,PWPX0982761090996,Oat Milk,Supermarket,15,19.58


In [31]:
# Visualize top missing products
if 'product_name' in missing_items.columns:
    fig = px.bar(
        top_missing_products.sort_values('times_missing', ascending=True),
        x='times_missing',
        y='product_name',
        orientation='h',
        color='category',
        title='Top 10 Most Frequently Missing Products',
        labels={
            'times_missing': get_label('times_missing'),
            'product_name': 'Product',
            'category': 'Product Category'
        },
        text='times_missing',
        hover_data=['price']
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500, showlegend=True)
    fig = apply_project_theme(fig)
    fig.show()

## 8. Collusion Analysis

In [32]:
# Detect collusion patterns
collusion_indicators = detect_collusion_patterns(orders, min_interactions=2, threshold_rate=40)

collusion_count = len(collusion_indicators)
print(f'Potential Collusion Cases Found: {collusion_count}')

if collusion_indicators:
    collusion_df = pd.DataFrame([{
        'driver_id': i.details['driver_id'],
        'customer_id': i.details['customer_id'],
        'interactions': i.details['interactions'],
        'missing_rate': i.details['missing_rate'],
        'items_missing': i.details['items_missing'],
        'risk_score': i.score
    } for i in collusion_indicators])
    
    print('\nTop Collusion Suspects:')
    display(collusion_df.head(10))

Potential Collusion Cases Found: 0


In [33]:
# Collusion network visualization (if cases exist)
if collusion_indicators and len(collusion_df) > 0:
    fig = px.scatter(
        collusion_df,
        x='interactions',
        y='missing_rate',
        size='items_missing',
        color='risk_score',
        hover_data=['driver_id', 'customer_id'],
        title='Potential Collusion Cases: Interactions vs Missing Rate',
        labels={
            'interactions': 'Number of Interactions',
            'missing_rate': get_label('missing_rate'),
            'risk_score': get_label('risk_score'),
            'items_missing': 'Items Missing'
        },
        color_continuous_scale='Reds'
    )
    fig.update_layout(height=450)
    fig = apply_project_theme(fig)
    fig.show()

## 9. Executive Summary

In [34]:
# Compile all key metrics for executive summary
print('=' * 70)
print('EXECUTIVE SUMMARY - FRAUD ANALYSIS')
print('=' * 70)

print('\n--- KEY FRAUD METRICS ---')
print(f'Overall Missing Rate: {overall_missing_rate:.2f}%')
print(f'Orders with Issues: {orders_with_missing:,} ({pct_orders_with_missing:.2f}%)')
print(f'Total Missing Items: {total_missing:,}')
print(f'Estimated Financial Loss: ${estimated_loss:,.2f}')

print('\n--- HIGH-RISK ENTITIES ---')
print(f'High-Risk Drivers: {high_risk_drivers_count}')
print(f'High-Risk Customers: {high_risk_customers_count}')
print(f'Worst Region: {worst_region} ({worst_region_rate:.2f}% missing rate)')
print(f'Potential Collusion Cases: {collusion_count}')

print('\n--- TEMPORAL INSIGHTS ---')
print(f"Trend Direction: {temporal_summary['trend']['direction'].upper()}")
print(f"Peak Month: {temporal_summary['trend']['peak_month']} ({temporal_summary['trend']['peak_rate']:.2f}%)")
print(f"Worst Day: {temporal_summary['patterns']['worst_day']}")
print(f"Worst Hour: {temporal_summary['patterns']['worst_hour']}:00")

print('\n--- PRODUCT INSIGHTS ---')
if category_stats is not None and len(category_stats) > 0:
    highest_loss_cat = category_stats.loc[category_stats['total_loss'].idxmax()]
    most_stolen_cat = category_stats.loc[category_stats['missing_count'].idxmax()]
    print(f"Most Stolen Category: {most_stolen_cat['category']} ({most_stolen_cat['missing_count']:,.0f} items)")
    print(f"Highest Loss Category: {highest_loss_cat['category']} (${highest_loss_cat['total_loss']:,.2f})")
else:
    print('Product category data not available')

EXECUTIVE SUMMARY - FRAUD ANALYSIS

--- KEY FRAUD METRICS ---
Overall Missing Rate: 1.63%
Orders with Issues: 1,502 (15.02%)
Total Missing Items: 1,657
Estimated Financial Loss: $97,978.30

--- HIGH-RISK ENTITIES ---
High-Risk Drivers: 419
High-Risk Customers: 15
Worst Region: Altamonte Springs (1.76% missing rate)
Potential Collusion Cases: 0

--- TEMPORAL INSIGHTS ---
Trend Direction: STABLE
Peak Month: 2023-08 (1.83%)
Worst Day: Monday
Worst Hour: 22:00

--- PRODUCT INSIGHTS ---
Most Stolen Category: Supermarket (1,393 items)
Highest Loss Category: Electronics ($133,398.60)


In [35]:
# Create executive summary table
from IPython.display import Markdown, display

# Format values for summary table
if category_stats is not None and len(category_stats) > 0:
    most_stolen_cat_name = category_stats.loc[category_stats['missing_count'].idxmax(), 'category']
else:
    most_stolen_cat_name = 'N/A'

summary_md = f"""
## Key Findings Summary Table

| Metric | Value |
|--------|-------|
| Overall Missing Rate | {overall_missing_rate:.2f}% |
| Orders with Issues | {orders_with_missing:,} ({pct_orders_with_missing:.2f}%) |
| Estimated Financial Loss | ${estimated_loss:,.2f} |
| High-Risk Drivers | {high_risk_drivers_count} |
| High-Risk Customers | {high_risk_customers_count} |
| Worst Region | {worst_region} ({worst_region_rate:.2f}%) |
| Potential Collusion Cases | {collusion_count} |
| Most Stolen Category | {most_stolen_cat_name} |
| Peak Fraud Month | {temporal_summary['trend']['peak_month']} |
| Trend Direction | {temporal_summary['trend']['direction'].upper()} |
"""

display(Markdown(summary_md))


## Key Findings Summary Table

| Metric | Value |
|--------|-------|
| Overall Missing Rate | 1.63% |
| Orders with Issues | 1,502 (15.02%) |
| Estimated Financial Loss | $97,978.30 |
| High-Risk Drivers | 419 |
| High-Risk Customers | 15 |
| Worst Region | Altamonte Springs (1.76%) |
| Potential Collusion Cases | 0 |
| Most Stolen Category | Supermarket |
| Peak Fraud Month | 2023-08 |
| Trend Direction | STABLE |


In [36]:
# Recommendations based on analysis
recommendations_md = f"""
## Recommendations

Based on the fraud analysis, we recommend the following actions:

### Immediate Actions (High Priority)

1. **Driver Monitoring Program**
   - Implement enhanced monitoring for the {high_risk_drivers_count} high-risk drivers identified
   - Require photo verification for deliveries by flagged drivers
   - Consider temporary suspension pending investigation for critical-risk cases

2. **Customer Verification**
   - Implement additional verification for the {high_risk_customers_count} high-risk customers
   - Require signature confirmation or photo proof for orders to flagged addresses
   - Review refund policies for repeat claimants

3. **Regional Focus: {worst_region}**
   - This region shows {worst_region_rate:.1f}% missing rate - significantly above average
   - Deploy additional oversight in this area
   - Consider route optimization to reduce delivery time

### Medium-Term Actions

4. **Collusion Investigation**
   - Investigate the {collusion_count} potential collusion cases identified
   - Implement random driver assignment for flagged customer addresses
   - Cross-reference patterns between drivers and customers

5. **Product Protection**
   - High-value categories require additional packaging/security
   - Consider GPS tracking for high-value deliveries
   - Implement tamper-evident packaging for frequently stolen items

6. **Temporal Controls**
   - Peak fraud month is {temporal_summary['trend']['peak_month']} - increase staffing/monitoring
   - {temporal_summary['patterns']['worst_day']} shows highest fraud - implement day-specific controls
   - Hour {temporal_summary['patterns']['worst_hour']}:00 is high-risk - consider delivery restrictions

### Long-Term Initiatives

7. **Technology Improvements**
   - Implement ML-based real-time fraud detection
   - Deploy delivery confirmation photos (before/after)
   - Consider smart locker installations in high-fraud areas

8. **Process Improvements**
   - Regular driver audits and training
   - Customer education on secure delivery practices
   - Establish fraud reporting hotline

### Expected Impact

Implementing these recommendations could reduce fraud losses by an estimated 30-50%, 
potentially saving **${estimated_loss * 0.4:,.2f}** annually.
"""

display(Markdown(recommendations_md))


## Recommendations

Based on the fraud analysis, we recommend the following actions:

### Immediate Actions (High Priority)

1. **Driver Monitoring Program**
   - Implement enhanced monitoring for the 419 high-risk drivers identified
   - Require photo verification for deliveries by flagged drivers
   - Consider temporary suspension pending investigation for critical-risk cases

2. **Customer Verification**
   - Implement additional verification for the 15 high-risk customers
   - Require signature confirmation or photo proof for orders to flagged addresses
   - Review refund policies for repeat claimants

3. **Regional Focus: Altamonte Springs**
   - This region shows 1.8% missing rate - significantly above average
   - Deploy additional oversight in this area
   - Consider route optimization to reduce delivery time

### Medium-Term Actions

4. **Collusion Investigation**
   - Investigate the 0 potential collusion cases identified
   - Implement random driver assignment for flagged customer addresses
   - Cross-reference patterns between drivers and customers

5. **Product Protection**
   - High-value categories require additional packaging/security
   - Consider GPS tracking for high-value deliveries
   - Implement tamper-evident packaging for frequently stolen items

6. **Temporal Controls**
   - Peak fraud month is 2023-08 - increase staffing/monitoring
   - Monday shows highest fraud - implement day-specific controls
   - Hour 22:00 is high-risk - consider delivery restrictions

### Long-Term Initiatives

7. **Technology Improvements**
   - Implement ML-based real-time fraud detection
   - Deploy delivery confirmation photos (before/after)
   - Consider smart locker installations in high-fraud areas

8. **Process Improvements**
   - Regular driver audits and training
   - Customer education on secure delivery practices
   - Establish fraud reporting hotline

### Expected Impact

Implementing these recommendations could reduce fraud losses by an estimated 30-50%, 
potentially saving **$39,191.32** annually.


In [37]:
# Final summary visualization - Risk Overview Dashboard
# Improved: Better layout, standardized colors, clear units

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Fraud Indicators by Type',
        'Regional Risk Distribution',
        'Driver Risk Categories',
        'Monthly Fraud Trend'
    ],
    specs=[
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'pie'}, {'type': 'scatter'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

# 1. Fraud indicators by type
fig.add_trace(
    go.Bar(
        x=indicator_counts['Type'],
        y=indicator_counts['Count'],
        marker_color=PROJECT_THEME['fraud_red'],
        name='Indicators',
        showlegend=False
    ),
    row=1, col=1
)

# 2. Regional risk - highlight worst region
regional_for_dash = regional.sort_values('missing_rate', ascending=False)
colors_regional = [PROJECT_THEME['fraud_red']] + [PROJECT_THEME['neutral_gray']] * (len(regional_for_dash) - 1)

fig.add_trace(
    go.Bar(
        x=regional_for_dash['region'],
        y=regional_for_dash['missing_rate'],
        marker_color=colors_regional,
        name='Missing Rate',
        showlegend=False
    ),
    row=1, col=2
)

# 3. Driver risk pie with standardized colors
# Order risk categories properly
risk_counts_ordered = risk_counts.set_index('Risk Category').loc[
    [cat for cat in PROJECT_THEME['risk_order'] if cat in risk_counts['Risk Category'].values]
].reset_index()

fig.add_trace(
    go.Pie(
        labels=risk_counts_ordered['Risk Category'],
        values=risk_counts_ordered['Count'],
        marker_colors=[PROJECT_THEME['risk_colors'][cat] for cat in risk_counts_ordered['Risk Category']],
        name='Risk Categories',
        showlegend=True
    ),
    row=2, col=1
)

# 4. Monthly trend
fig.add_trace(
    go.Scatter(
        x=monthly['month_name'],
        y=monthly['missing_rate'],
        mode='lines+markers',
        marker_color=PROJECT_THEME['fraud_red'],
        marker_size=8,
        line=dict(width=3, color=PROJECT_THEME['fraud_red']),
        name='Missing Rate Trend',
        showlegend=False
    ),
    row=2, col=2
)

# Update axes with proper labels and units
fig.update_xaxes(title_text='Indicator Type', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)

fig.update_xaxes(title_text='Region', row=1, col=2, tickangle=-45)
fig.update_yaxes(title_text='Missing Rate (%)', row=1, col=2)

fig.update_xaxes(title_text='Month', row=2, col=2)
fig.update_yaxes(title_text='Missing Rate (%)', row=2, col=2)

fig.update_layout(
    title_text='Fraud Analysis Dashboard - Executive Summary View',
    height=800,
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.08,
        xanchor='center',
        x=0.5
    )
)

fig = apply_project_theme(fig)
fig.show()

In [38]:
print('\n' + '=' * 70)
print('FRAUD ANALYSIS COMPLETE')
print('=' * 70)
print(f'\nAnalysis Date: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Data Period: {orders["order_date"].min()} to {orders["order_date"].max()}')
print(f'Total Records Analyzed: {total_orders:,} orders')
print('\nNext Steps:')
print('  - Review high-risk entities in detail')
print('  - Implement recommended preventive measures')
print('  - Set up monitoring dashboard (Notebook 04)')
print('  - Train ML models for real-time detection')


FRAUD ANALYSIS COMPLETE

Analysis Date: 2026-01-09 22:07:15
Data Period: 2023-01-01 to 2023-12-31
Total Records Analyzed: 10,000 orders

Next Steps:
  - Review high-risk entities in detail
  - Implement recommended preventive measures
  - Set up monitoring dashboard (Notebook 04)
  - Train ML models for real-time detection
